# Baseball Lab 7: Probability models and expected values

In today's lab you'll get practice:

1. Comparing observed data to a probability model
2. Running a randomization test to see if the observed data is significantly different from the expected data based on a probability model
3. Creating a run expectancy matrix that shows the expected number of runs scored in a baseball game given the current game state (e.g., how many runs are expected to score when there are 2 outs and bases loaded, etc.). 


#### Deadline

This assignment is due **Sunday March 29th at 11pm**. You can turn in the assignment up to 24 hours late for 90% credit (after that, the homework will only be accepted with a Dean's Extension). Directly sharing answers is not okay, but discussing problems with the course staff or with other students is encouraged. Refer to the policies page to learn more about how to learn cooperatively. You should start early so that you have time to get help if you're stuck. If you have questions, please post them to [Ed Discussion](https://edstem.org/us/courses/89102/discussion) (and answer others' questions too). 


## Getting started - downloading the data

In order to complete this lab, it is necessary to download a few files. Please run the code below **only once** to download data needed to complete the lab. To run the code, click in the cell below and press the play button (or press shift-enter). 


In [40]:
# Please run this code once to download the files you will need to complete the homework 

import YData_baseball


def download_retro_data(year):
    
    import os.path
    retro_file_name = str(year) + "plays.zip"
    if not os.path.isfile(retro_file_name):
        import requests
        retro_url = "https://www.retrosheet.org/downloads/plays/" + retro_file_name
        r = requests.get(retro_url )
        with open(retro_file_name, "wb") as f:
            f.write(r.content)
    else:
        print("File already downloaded, skipping download step.")


download_retro_data(2025)

File already downloaded, skipping download step.


# Part 0: Quote and reaction to Astroball chapter 7 (5 points)

Please find an interesting quote from chapter 7 of Astroball and then write a ~one paragraph reaction to the quote below.

*Quote:*  ...

Reaction: ... 

In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Part 1: Comparing observed data to a probability model 

In class we discussed how probability models can be used to calculate the probability of particular events occurring. For example, we discussed how the binomial distribution could be used to model the probability of [Koji Uehara](https://www.baseball-reference.com/players/u/ueharko01.shtml) striking out a batter on 5 pitches. Let's now explore the binomial probability model further by comparing the number of hits Aaron Judge gets in a game to the binomial distribution! 

Recall that the binomial distribution models the number of successes in a fixed number of independent Bernoulli trials, where each trial has the same probability of success. The probability mass function (PMF) of the binomial distribution has the form:  $$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$ where:
- $n$ is the number of trials
- $k$ is the number of successes
- $p$ is the probability of success. 

In this case, we will model the number of hits Aaron Judge gets in a game as a binomial distribution when he had 4 at-bats assuming he has a fixed probability of getting a hit at each at-bat. We can then compare the observed number of hits he got in each game he had 4 at-bats to the expected number of hits based on the binomial distribution. This will help us assess whether it is reasonable to model each at-bat Aaron Judge has is independent from each other at-bats with the same probability of a hit occurring (which are critical assumptions of the binomial model). 

For our analyses, we will use the retrosheet play-by-play data for the 2025 season which is loaded below. 


In [42]:
retro_data = pd.read_csv("2025plays.zip", low_memory=False)
retro_data.head()

,gid,event,inning,top_bot,vis_home,site,batteam,pitteam,score_v,score_h,...,pn,umphome,ump1b,ump2b,ump3b,umplf,umprf,date,gametype,pbp
0,CHN202503180,43/G34,1,0,0,TOK01,LAN,CHN,0,0,...,1,millb901,estam901,barrl901,libkj901,NaN,NaN,20250318,regular,full
1,CHN202503180,3/P34,1,0,0,TOK01,LAN,CHN,0,0,...,2,millb901,estam901,barrl901,libkj901,NaN,NaN,20250318,regular,full
2,CHN202503180,K,1,0,0,TOK01,LAN,CHN,0,0,...,3,millb901,estam901,barrl901,libkj901,NaN,NaN,20250318,regular,full
3,CHN202503180,W,1,1,1,TOK01,CHN,LAN,0,0,...,4,millb901,estam901,barrl901,libkj901,NaN,NaN,20250318,regular,full
4,CHN202503180,6/L6MD,1,1,1,TOK01,CHN,LAN,0,0,...,5,millb901,estam901,barrl901,libkj901,NaN,NaN,20250318,regular,full


## 1.1: Extracting Aaron Judge's data

To start, let's extract the data for Aaron Judge from the retrosheet play-by-play data. 

**Exercise 1.1 (6 points):** Please create a DataFrame called `judge_data` that contains only the plays where Aaron Judge was the batter, and that were plate appearances (i.e., where the batter had a chance to hit the ball). Also, please filter the data to only include regular season games (i.e., not playoff or exhibition games), and add a variable called `h` that contains the number of hits Aaron Judge had. To do this, please complete the following steps: 

1. Filter the data to only include plays where Aaron Judge was the batter.
2. Filter the data to only include plays that were plate appearances (i.e., where the batter had a chance to hit the ball).
3. Filter the data to only include regular season games (i.e., not playoff or exhibition games).
4. Add a column called `h` that contains the number of hits the batter got in each play. The number of hits is the sum of the number of singles, doubles, triples, and home runs.
5. Print out the first 5 rows of the `judge_data` DataFrame to verify that the data is correct. 


## 1.2: Calculating the number of hits Aaron Judge had in each game

Let's now calculate the number of at-bats and hits Aaron Judge had in each game (recall that at-bats are the number of times a player comes to bat, excluding walks, hit by pitches, and sacrifices). 


**Exercise 1.2 (6 points):** Please create a DataFrame called `judge_hits` where each row corresponds to a given game ID (gid) that Aaron Judge played in and the columns of the DataFrame are: 

1. `gid`: the game ID  which should be the row Index of this `judge_hits` DataFrame
2. `ab`: the number of at-bats Aaron Judge had in a given game  
3. `h`: the number of hits Aaron Judge had in a given game  

Please print out the first 5 rows of the `judge_hits` DataFrame to show that the data is correct. 


## 1.3: Sanity check - calculating Judge's batting average

Before we proceed, let's do a quick sanity check to make sure that the data we have is correct by calculating Aaron Judge's batting average for the games he played in. Recall that the batting average is calculated as the number of hits divided by the number of at-bats. We will also use this batting average when we create our binomial distribution below.

**Exercise 1.3 (3 points):** Please calculate Aaron Judge's batting average for the games he played in and save it to the name `batting_average`. Then print it out to show your work. When you print it out, please display this value to 3 decimal places, although do not round the actual value stored in the batting average variable. 

Also, go to [baseball reference](https://www.baseball-reference.com/) and check that this matches the batting average listed for Aaron Judge. Do not proceed to the following exercise until you have verified that the batting average is correct!


## 1.4: Displaying the number of at-bats Judge typically had 

Let's now exam how many at-bats Judge typically had in a game.

**Exercise 1.4 (4 points):** Please display a DataFrame where each row corresponds to a different number of at-bats, and the column displays the frequency of that number of at-bats that Judge had; i.e., the first row should correspond to the number of games Judge at 1 at-bat, and the column value should show the number of games he had 1 at-bat games he had, etc. 

Also, report the number of games Judge had 4 at-bats in a game in the answer section below. 


**Answer**



## 1.5: Calculating the frequency of hits in 4 at-bats games

For the rest of the analyses in this section, let's focus on games where Judge had 4 at-bats, and let's examine how many hits he had in those games. To start, let's visualize frequency of hits in games where Judge had 4 at-bats. 

**Exercise 1.5 (4 points):** Please create a DataFrame called `hit_freq_4ab` where each row corresponds to the number of hits Judge had in a game (in games where he had 4 at-bats), and the column displays the count of how many times he had that number of hits;   i.e., the first row should correspond to the number of games Judge had 0 hits (in a games where he had 4 at-bats), and the column value should show the number of games he had 0 hits (in a game where he had 4 at-bats, etc).

Hint: Filtering the `judge_hits` DataFrame to only include games where Judge had 4 at-bats could be a good first step.


## 1.6 Visualization of the frequency of hits in games where Judge had 4 at-bats

Let's now visualize the proportion of games where Judge had a particular number of hits (in games where he had 4 at-bats). 

**Exercise 1.6 (3 points):** Please create a Series called `observed_frequencies` where the index is the number of hits Judge had in a game (in games where he had 4 at-bats), and the values are the proportion of games where Judge had that many hits. Then, please create a bar plot that shows the proportion of games where Judge had different numbers of hits (for games where he had 4 at-bats). The x-axis should be the number of hits, and the y-axis should be the proportion of games where he had that many hits. 


## 1.7: Getting predicted frequencies of hits from a binomial distribution

Let's now assess how many hits Judge would be expected to get in a game where he had 4 at-bats based on the binomial distribution. To do this, we will use the batting average we calculated above as the probability of success in each at-bat, and we will assume that each at-bat is independent. For modeling Judge's number of hits in a game where he had 4 at-bats, we will use the binomial distribution with `n=4` and `p=batting_avg`. As we discussed in class and at the start of the part of the homework, the equation for the binomial probability mass function (PMF) is:  $$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$.

**Exercise 1.7 (5 points):** Below we import the `scipy.stats.binom` module so that we can use the `binom.pdf()` function from the scipy package, which allows us to calculate the binomial probability mass function (PMF). Please complete the code below to create a variable called `binomial_distribution` that has the binomial PMF where:

- `k`: is a sequence of integers from 0 to 4. Hint: the `range()` function could be useful for creating this sequence.
- `n`: is the number of at-bats (which is 4).
- `p`: is the proportion of times we would expect Judge to get a hit in a game, which is the batting average we calculated above. 


Once you have created this `binomial_distribution` variable, display it as a bar plot where the x-axis is the number of hits, and the y-axis is the probability of getting that many hits. In the answer section, report whether this binomial distribution looks similar to the observed frequencies of hits in games where Judge had 4 at-bats. 


In [43]:
from scipy.stats import binom






**Answer**




## 1.8: Displaying the observed and expected frequencies of hits next to each other

Comparing the observed frequencies with which Judge got a particular number of hits to the predicted number of hits based on the binomial distribution is a good way to see how well the binomial distribution fits the observed data. However, it is hard to make this visual comparison across two different plots. Instead it is better to display the observed and expected frequencies of hits next to each other in a single plot so let's do that now!

**Exercise 1.8 (6 points):** Please create a bar plot that displays the observed frequencies of hits in games where Judge had 4 at-bats in one color, next to the expected frequencies of hits based on the binomial distribution in another color; i.e., the x-axis should be the number of hits, and the y-axis should be the frequency of that many hits that are observed (in one color) and that are predicted by the binomial distribution (in another color). 

In order to create this side-by-side bar plot, you will need to use the `plt.bar()` function twice, once for the observed frequencies and once for the expected frequencies. In particular, please complete the following steps to create this plot.

1. Create a sequence of integers from 0 to 4 in a variabled named `x`, which will be used as the x-axis values for the bar plot. We have done this for you below using `np.arange(5)`. 

2. Create a variable called `width` and set it to 0.4, which will be used to adjust the width of the bars so that they are next to each other (we have also done this for you).

3. Create the first bar plot using `plt.bar()` where:
   - The x-axis values are `x - width/2` (this will shift the bars to the left so they are next to each other).
   - The y-axis values are the observed frequencies of hits.
   - The width of the bars is set to `width`.
   - The label for this bar plot is "Observed".   

<p>

4. Create the second bar plot using `plt.bar()` where:
   - The x-axis values are `x + width/2` (this will shift the bars to the right so they are next to each other).
   - The y-axis values are the expected frequencies of hits based on the binomial distribution.
   - The width of the bars is set to `width`.
   - The label for this bar plot is "Binomial".

<p></p>

5. Finally, add labels for the x-axis and y-axis, a title for the plot, and a legend to distinguish between the observed and expected frequencies. 


In the answer section report once again whether the observed frequencies of hits in games where Judge had 4 at-bats looks similar to the expected frequencies based on the binomial distribution. 


In [44]:

x = np.arange(5)
width = 0.4



# Create the side-by-side bar plots below









**Answer**





# Part 2: Testing agreement between a probability model and observed data

In the previous section we visually compared the observed frequencies of hits in games where Judge had 4 at-bats to the predicted frequencies based on the binomial distribution and saw that they looked similar (at least they should have :). Let's now explore this in a more quantitative manner by running a hypothesis test. 

Our hypothesis test will examine whether the frequencies of the number of hits Aaron Judge has 4 at-bat sequences is significantly different from the expected frequencies based on the binomial distribution. If they are different, this will indicate that the assumptions of the binomial model, that each at-bat is independent with the same probability of occurring, have been violated. 

To compare the binomial distribution to the observed data, we will use Total Variation Distance (TVD) test statistic which is defined as: $$TVD(P, Q) = \frac{1}{2} \sum_{i} |P(x_i) - Q(x_i)|$$ where $P$ and $Q$ are two probability distributions. In our case, $P$ will be the observed frequencies of hits in games where Judge had 4 at-bats, and $Q$ will be the expected frequencies based on the binomial distribution. 


We can then use the following steps to run a hypothesis test:

1. Calculate the total variation distance (TVD) between the observed frequencies and the expected frequencies based on the binomial distribution. This shows us how far the actual data differs from the theoretical probabilities. 

2. Create a "null distribution" of TVD values by randomly generating frequencies of hits based on the binomial distribution and comparing these to expected frequencies based on the binomial distribution. This will allow us assess the typical variation that in observed frequencies we would expect from the binomial distribution. We will do this by simulating a large number of random samples from the binomial distribution and calculating the TVD for each sample. 

3. Compare the observed TVD (from the real data) to the null distribution of TVD values. This will allow us to see if the observed TVD is significantly different from the expected TVD statistics based on the binomial distribution. If the observed TVD is much larger than the typical variation we would expect from the binomial distribution, this will indicate that the assumption that each at-bat is independent with the same probability does not appear to model the data well. 

Note: since this is a data science class we will focus on using simulation methods to generate the null distribution of TVD values, rather than using a theoretical distribution to generate the null distribution. Another approach, would be to use chi-square test to compare the observed and expected frequencies (i.e., the Chi-square Goodness-of-Fit test). To learn more about that, take additional statistics classes!


## 2.1: Calculating the total variation distance (TVD)

To start on our hypothesis test let's write a function that calculates the total variation distance (TVD) between two probability distributions. 

**Exercise 2.1 (4 points):** Please create a function called `tvd()` that takes two sequences (e.g., ndarrays) as input and returns the total variation distance between them. The function should take two arguments: `p` and `q`, which represent probability distributions and/or observed proportions. The function should return the total variation distance between `p` and `q`. 

Once you have written the function, calculate the total variation distance between the observed frequencies of hits in games where Judge had 4 at-bats  and the expected frequencies based on the binomial distribution (i.e., calculate the total variation distance between `observed_frequencies` and `binomial_distribution`). Save this value to a variable called `observed_stat` and print out the value to show your results.


In [45]:
def tvd(p, q):
    ...
    

## 2.2: Generating random samples from the binomial distribution

Let's now use the `np.random.binomial()` function to generate random samples from the binomial distribution. We will use this to simulate the number of hits Judge would get in a game where he had 4 at-bats based on the binomial distribution. 

**Exercise 2.2 (4 points):** Please create a variable called `sim_judge_hit_counts` that contains a random sample of the number of hits Judge would get in a game where he had 4 at-bats based on the binomial distribution; i.e., this will simulate the number of hits (numbers between 0 and 4) for all 72 games. 

To do this, use the `np.random.binomial()` function with the following parameters:

1.  `n`: the number of at-bats (which is 4).
2.  `p`: the probability of getting a hit in each at-bat, which is the batting average we calculated above.
3.  `size`: the number of games we want to simulate (which is 72, the number of games Judge had 4 at-bats in). 

Please generate one random sample that simulates how many hits Judge would get in a game where he had 4 at-bats in the 72 games he played in and save this to the variable `sim_judge_hit_counts`. 


In [46]:
# set the random seed for reproducibility. Do not change the seed value.
np.random.seed(1)  






## 2.3 Calculating the proportion of hit counts in the simulated data

Let's now calculate the proportion of times Judge got a particular number of hits in the simulated data. 

**Exercise 2.3 (4 points):** Please calculate the number of times Judge got a particular number of hits in the simulated data (i.e., `sim_judge_hit_counts`). To do this, use the `np.bincount()` function to count the number of times each number of hits occurred in the simulated data. The `np.bincount()` function takes the following arguments:

1. `sim_judge_hit_counts`: the simulated data that contains the number of hits Judge got in each of the 72 games where he had 4 at-bats.

2. `minlength`: the minimum length of the output array, which should be the number of at-bats (which is 4 + 1). This will ensure that the output array has a length of 5, corresponding to the number of hits from 0 to 4. 

Then divide the counts by the total number of games (72) to convert these counts to proportions. Save this to a variable called `sim_judge_freq` and print out these values to show your work. 


## 2.4: Visualizing the simulated frequencies of hits 

Let's now visualize the simulated frequencies of hits in games where Judge had 4 at-bats, and also calculate the total variation distance between these simulated frequencies and the proportions predicted by the binomial distribution (i.e., the `binomial_distribution`). 

**Exercise 2.4 (4 points):** Please create a bar plot that shows the simulated frequencies of hits in games where Judge had 4 at-bats. The x-axis should be the number of hits, and the y-axis should be the proportion of games where he had that many hits. Then calculate the total variation distance between these simulated frequencies and the proportions of hits in the observed data (i.e., the `observed_frequencies`) and print it out to show your results. 


## 2.5: A function to generate random TVD values

Let's now write a function that can generate a random TVD value based on the binomial distribution that consolidates exercises 2.2-2.4 above. This will allow us to easily generate random TVD values for later analyses. 

**Exercise 2.5 (5 points):** Please create a function called `get_random_judge_tvd_value()` that takes no arguments and returns a random TVD value based on the binomial distribution. The function should do the following:

1. Generate a random sample of the number of hits Judge would get in a game where he had 4 at-bats over the 72 games he played with 4 at-bats (i.e., the values you calculated in exercise 2.2).

2. Calculate the proportion of times Judge got a particular number of hits in the simulated data (i.e., the values you calculated in exercise 2.3).

3. Calculate the total variation distance between the simulated frequencies and the expected frequencies based on the binomial distribution (i.e., the values you calculated in exercise 2.4).

Once you have created this function, use the code below to generate a random TVD value in order to see that the function is working correctly.


In [47]:
def get_random_judge_tvd_value():

    ...


# set the random seed for reproducibility. Do not change the seed value.
np.random.seed(100)  

get_random_judge_tvd_value()
    

## 2.6 Generating a null distribution of TVD values

In the previous step we created a function that can generate a random TVD value that we might expect to see from Judge over 72 games if the data truly came from a binomial distribution. We can now use this function to create a "null distribution of 10,000 random TVD values which will allow us to assess how much variation we would expect if binomial model was correct. 

**Exercise 2.6 (4 points):** Please create a list called `null_dist` that contains 10,000 random TVD values generated using the `get_random_judge_tvd_value()` function you created in the previous step. You can use a for loop to generate these values. Once you have created this list, convert it to a NumPy array and print out the first 5 values to show your work. 





In [48]:
null_dist = []
    
# use a for loop to add randomly generated TVD valus to the null distribution list...






## 2.7 Visualizing the null distribution of TVD values

Let's now visualize this null distribution of TVD values to see how much variation we would expect if the binomial model was correct and compare this to our observed TVD value from the real data. 

**Exercise 2.7 (5 points):** Please create a histogram of the null distribution of TVD values you created in the previous step. The x-axis should be the TVD values, and the y-axis should be the frequency of each TVD value. Then add a vertical line to the histogram that shows the observed TVD value from the real data (i.e., `observed_stat`) using the `plt.axvline()` function. 

Because the TVD statistic is larger when the observed data is more different from the expected data, we will use a right-tailed test to assess whether the observed TVD value is significantly different from the expected TVD values based on the binomial distribution. From looking at this plot, does it look like the observed TVD value is significantly different from the TVD values that would be expected based on the binomial model? Please answer this question in the answer section below. 


**Answer**






## 2.8 Calculating a p-value

A p-value is the probability of observing a test statistic as extreme as the one we observed (or more extreme) under the null hypothesis. In our case, the null hypothesis is that the observed frequencies of hits in games where Judge had 4 at-bats are not significantly different from the expected frequencies based on the binomial distribution. 

**Exercise 2.8 (4 points):** Please calculate the p-value for the observed TVD value using the null distribution of TVD values you created in the previous step. The p-value should be the proportion of values in the null distribution that are greater than or equal to the observed TVD value. Save this value to a variable called `p_value` and print it out to show your work. 


## 2.9 Making a decision

Typically we would reject the null hypothesis if the p-value is less than 0.05, which would indicate that the observed frequencies of hits in games where Judge had 4 at-bats are significantly different from the expected frequencies based on the binomial distribution. 

**Exercise 2.9 (4 points):** Based on the p-value you calculated in the previous step, would you reject or fail to reject the null hypothesis? Please answer this question in the answer section below and describe that this says about using the binomial distribution as a model for Aaron Judge's hitting performance in games where he had 4 at-bats. 


**Answer**








## Bonus: Using the Chi-square goodness of fit test (0 points)

For those who are interested, one can also use the Chi-square goodness of fit test to compare the observed frequencies to the expected frequencies from the binomial distribution using the code below. Below I load the `scipy.stats.chisquare` function which allows us to run the Chi-square goodness of fit test. 

The hypotheses for this test are the same as above, namely, the null hypothesis for this test is that the observed frequencies are not significantly different from the expected frequencies based on the binomial distribution. As usual, if the p-value is less than 0.05, we can reject the null hypothesis and conclude that the observed frequencies are significantly different from the expected frequencies based on the binomial distribution. 

If you are interested in running the Chi-square goodness of fit test, you can convert the `sim_judge_freq` and `binomial_distribution` to the number of hits (i.e., multiplying these proportions by 72) and then run the Chi-square goodness of fit test using the `scipy.stats.chisquare()` function (please use Google, chatGPT, etc. to learn more about the `chisquare()` function).  


In [49]:
from scipy.stats import chisquare









# Part 3 Creating the run expectancy matrix

As we discussed, the expected value of a random variable is the average value of that variable will have (i.e., it is the sum of the product of each possible value of the variable and its probability). The expected value is a useful concept in baseball since it can be used to quantify the average outcome of a particular game situation 

In this set of exercises you will calculate the "run expectancy matrix" which is a table that shows the expected number of runs scored in a baseball game given the current game state. The run expectancy matrix is a useful tool for understanding how the current game state (e.g., the number of outs, the state of the base runners, etc.) affects the expected number of runs scored by the end of the inning. More formally, the run expectancy matrix is defined as: 


**Rows**: runners on base states (8 states total):
  - bases empty:  000
  - runner on first:  001
  - runner on second: 010
  - runner on third:  100
  - runners on first and second:  011
  - runners on first and third:   101
  - runners on second and third:  110
  - runners on all three bases:   111

<p>
  
**Columns**: outs states (3 states total):
  - 0 outs
  - 1 out
  - 2 outs

<p>

**Fill values**: expected number of runs scored by the end of the inning given the current game state. 


In the following exercises, we will use the 2025 retrosheet play-by-play data to create an approximate run expectancy matrix. We will start by focusing on the big picture where you will try to figure out the steps needed to compute the run expectancy matrix. You will then go ahead and calculate an approximate run expectancy matrix. 


## 3.1 Determining the steps needed to calculate the run expectancy matrix

As we discussed in lab 4, when manipulating data to create a specific desired DataFrame, it is very useful to think about the steps that are needed before doing any coding. As we also discussed in that lab, it can be useful to think both in the forward direction, starting with the initial data you have, and also in the backward direction, starting with the final table you want to generate.

Let's again think about the steps in the backward direction. In particular, suppose that for the *second to last* step you had a DataFrame called `remaining_runs_df` where each row of the DataFrame corresponded to a plate appearance, and columns of the DataFrame were: 

1. `base_runner_state`: A column of strings that has the combination of base runners on base at the start of the plate appearance (e.g., "000" for bases empty, "100" for a runner on third, etc.). 
2. `outs_pre`: A column of numbers that has the number of outs at the start of the plate appearance (0, 1, or 2).
3. `runs_remaining`:  A column of numbers that has the number of runs that have not yet been scored in the inning at the start of the plate appearance (i.e., the final score in the inning minus the current score). 

If you had this DataFrame, what DataFrame method could you use to create the run expectancy matrix? 

**Exercise 3.1 (4 points):** Please write down the code that would convert the `remaining_runs_df` DataFrame to the run expectancy matrix in the answer section below. 


**Answer**

```

Fill in your code here


```


## 3.2 Selecting relevant rows

Now that we have a sense of what our end results look like, we can think about the steps needed to transform the `retro_data` DataFrane into the `remaining_runs_df` DataFrame we will need to get the final run expectancy matrix. 

**Exercise 3.2 (4 points):** To make our data easier to deal with, as a first step let's create a DataFrame called `retro_data_smaller` that only has the columns that you will need for subsequent analyses. In particular, please select the following columns from the `retro_data` DataFrame: 

1. The column that uniquely identifies each game
2. The column that indicates the inning of the play
3. The column that indicates the team that is batting
4. The column that indicates the number of outs at the start of the play
5. The two columns that indicates the score of the home team and the visiting team at the start of the play
6. The three columns that indicate the state of the base runners at the start of the play

Please also for this and all subsequent exercises, please print out the first 5 rows of the DataFrame you create (in this case the `retro_data_smaller`) to show that the data is correct. 

Hint: A data codebook that explains what each column in the `retro_data` DataFrame is can be found at https://retrosheet.org/downloads/plays.html 


In [50]:
retro_data_smaller = retro_data.copy()






## 3.3 Creating the relevant columns

Let's now create a DataFrame called `runner_state_df` add a column to the `retro_data_smaller` DataFrame indicating the state of the base runners at the start of the play. As we discussed, we can represent the state of the base runners using a three digit binary number where each digit indicates whether there is a runner on a particular base (1 for a runner on that base, 0 for no runner on that base). 

**Exercise 3.3 (4 points):** Please create a DataFrame called `runner_state_df` that has a column called `base_runner_state` which indicates the state of the base runners for all plays in the `retro_data_smaller` DataFrame. The base runner state should use a three digit binary number where the left most digit indicates if there is a runner on third base, the middle digit indicates whether a runner is on second base, and the right most digit indicates whether a runner is on first base. For example, the number 100 would indicate a runner on 3rd base, while the digit 011 would indicate runners on first and second base. 

To make this a little easier for you, I have created a DataFrame below called `bases` which has the state of the base runners at the start of the play in three separate columns. You can use this DataFrame to create the `base_runner_state` column in the `runner_state_df` DataFrame (although make sure you understand what this code is doing). Also, once you have added the `base_runner_state` column to the `runner_state_df` DataFrame, please remove the original three columns that had the names of the base runners on first, second and third base.


In [51]:
bases = retro_data_smaller[['br3_pre', 'br2_pre', 'br1_pre']].notna().astype(int).astype(str)
runner_state_df = retro_data_smaller.copy()



# Create the base_runner_state column by concatenating the bases together, and then remove the original 3 columns indicating the names of the base runner





## 3.4 Create total score and half-inning ID

Two useful columns to create will be the total game score after each play (i.e., the sum of the scores of both teams), and a unique identifier for each half-inning of every game. Let's create a DataFrame that has these columns along with the base runner state and the number of outs at the start of the play. 


**Exercise 3.4 (5 points):** Please create a DataFrame called `total_score_df` that has the following columns:

1. `half_inn_id`: A unique ID for each half-inning of every baseball game. This can be created by concatenating the game ID, the inning number, and the batting team.
2. `base_runner_state`: The column you created in the previous step that indicates the state of the base runners at the start of the play. 
3. `pre_outs`: A column that has the number of outs at the start of the play
4. `tot_score`: A column that has the total game score after each play, which is the sum of the scores of both teams at the start of the play. 

As always, print out the first few rows to show your work. 


In [52]:
total_score_df = pd.DataFrame() 








## 3.5 Adding the score at the end of all unique innings

As we discussed above, in order to create the run expectancy matrix, we need to calculate the number of runs that have not yet been scored in each half-inning. To do this, we will first create a DataFrame that has the score at the end of each unique half-inning, and once we have this DataFrame, we can then calculate the number of runs that have not yet been scored in each half-inning by subtracting the current score from the end of the inning score. Let's start this process by first just adding a column that has the score at the end of each unique half-inning. 

**Exercise 3.5 (5 points):** Please create a DataFrame called `max_inning_scores_df` that has the same columns as the `total_score_df` DataFrame, but only has the score at the end of each unique half-inning.

Hint: To do this it could be useful to create a separate DataFrame that has one row for each unique half-inning, and has two columns which are: 1) the unique half-inning ID and 2) the total score at the end of that half-inning. You can then merge this DataFrame on to the `total_score_df` DataFrame to create a DataFrame that has the original columns along with the score at the end of each unique half-inning. 


## 3.6  Calculate the number of runs that have yet to be scored in each half-inning

Let's now create a DataFrame called `remaining_runs_df` that has the number of runs that have not yet been scored in each half-inning. 

**Exercise 3.6 (5 points):** Please create a DataFrame called `remaining_runs_df` that has the following columns:

1. `base_runner_state`: The column you created in the previous step that indicates the state of the base runners at the start of the play.

2. `outs_pre`: The column you created in the previous step that indicates the number of outs at the start of the play.

3. `runs_remaining`: A column that has the number of runs that have not yet been scored in the inning at the start of the play. This can be calculated by subtracting the current score at the start of the half-inning (i.e., `tot_score`) from the score at the end of the half-inning (i.e., `max_inn_score`). 


## 3.7 Create run expectancy matrix

We are now ready to create the run expectancy matrix! 

**Exercise 3.7 (5 points):** Please create a run expectancy matrix using the `remaining_runs_df` DataFrame you created in the previous step. You can use the code you wrote in the answer section of exercise 3.1 to do this. Once you have created the run expectancy matrix, please print it out to show your work. 

In the answer section report how many runs is a team expected to score in the following situations: 
1. At the start of an inning
2. When there are no outs and the bases are loaded
3. When there are 2 outs and bases loaded. 

**Answer**

1. 
  
2. 

3. 



## 3.8 Visualizing the run expectancy matrix

Let's now visualize the run expectancy matrix we created in the previous step. 

**Exercise 3.8 (5 points):** Please create a heatmap of the run expectancy matrix you created in the previous step. The x-axis should be the number of outs, and the y-axis should be the state of the base runners. The values in the heatmap should be the expected number of runs scored by the end of the inning given the current game state. 

In the answer section report whether it is better to have a runner on third base, or runners on first and second base. 


**Answer**







# 4. Reflection (3 points)

Please fill out the lab 7 reflection on Canvas to let us know how this lab, and the class overall, is going for you. 



# 5.  Submitting your work

Once you're finished filling in and running all cells, you should submit your assignment as a pdf on Gradescope. You can access Gradescope through Canvas on the left-side of the class home page. The problems in each lab assignment are numbered. When submitting on Gradescope, please **make sure to select the correct pages of your pdf that correspond to each problem**. Failure to mark pages correctly **will result in points being deducted** from your score.

To convert this Jupyter notebook document to a pdf please run the code in the cell below. This should produce a pdf document which should appear in the files tab on the left (you might need to refresh the files tab to see this file). You can then right click on this file (command click on a mac), to download this pdf document which you can upload to Gradescope. 

Please be sure to check that all the code and output are visible before submitting your pdf to Gradescope as **points will be deducted for missing code and output that is not visible** since we will not be able to grade this.

In [53]:
%%capture

!quarto render lab_07.ipynb --cache-refresh --to pdf 


#### Alternative submission instructions

If converting your Jupyter notebook to a pdf using the command in the cell above does not work, an alternative way to convert your Jupyter notebook is:

1.  Go to "File" at the top-left of your Jupyter Notebook
2.  Under "Download as" (or "Save and Export Notebook As...") and select "HTML (.html)"
3.  After the .html has downloaded, open it and then select "File" and "Print" (note you will not actually be printing)
4.  From the print window, select the option to save as a .pdf
